# EpiGraph - multi-seed robustness study (Kaggle)

Requirements: **GPU** accelerator, **Internet** enabled, and an input dataset containing the two `.pkl` files.
Cells are run top to bottom. For the long run, **Save Version -> Save & Run All (Commit)** keeps the notebook running in the background.

Contents
1. Evaluates the authors' 10 checkpoints with 4-decimal metrics + 95% bootstrap CI over test antigens.
2. **Experiment A** - retrains the full 10-fold ensemble for several *weight-initialisation* seeds (fold split fixed at the paper's seed 30).
3. **Experiment B** - retrains with several *fold-split* seeds (split seed and init seed both change).
4. Aggregates everything into mean +- std tables (`results/summary.md`).

One full ensemble = 10 folds x 50 epochs ~ 33 min on a Kaggle GPU. The notebook is resumable:
runs that already have a `metrics.json` are skipped.

In [ ]:
# ---- configuration ---------------------------------------------------------
INIT_SEEDS  = [1, 2, 3, 4, 5]   # experiment A: torch seeds, split seed stays BASE_SPLIT_SEED
SPLIT_SEEDS = [31, 32, 33]      # experiment B: split seed = torch seed = value ([] to skip)
BASE_SPLIT_SEED = 30            # upstream default (--seed)
EPOCHS = 50                     # upstream default
KEEP_CHECKPOINTS = False        # True keeps ~120 MB of .pt files per run in the notebook output

In [ ]:
# ---- 1. clone upstream repo, copy data, install --------------------------------
import os, glob, shutil, subprocess, sys
W = "/kaggle/working"
os.chdir(W)
if not os.path.exists("EpiGraph"):
    subprocess.run(["git", "clone", "https://github.com/sj584/EpiGraph"], check=True)
pkls = glob.glob("/kaggle/input/**/*.pkl", recursive=True)
assert pkls, "No .pkl found - add the dataset with epitope3d_*_data.pkl as notebook input"
for p in pkls:
    shutil.copy(p, f"{W}/EpiGraph/")
os.makedirs(f"{W}/src", exist_ok=True)
print(pkls)

In [ ]:
!pip install -q torch_geometric

In [ ]:
%%writefile /kaggle/working/src/patch_upstream.py
"""Compatibility patches for the upstream EpiGraph code (https://github.com/sj584/EpiGraph).

The upstream code was written against an older torch_geometric. With a recent version
(e.g. on Kaggle) three small things break. This script fixes them in place and is
idempotent (safe to run many times).

    python src/patch_upstream.py --upstream /kaggle/working/EpiGraph

1. model.py    : `data.node_attrs` collides with a method of the new PyG `Data` class
                 -> use `data['node_attrs']`.
2. evaluate.py : old checkpoints store GATConv weights as `lin_src` / `lin_dst`,
                 new PyG uses a single `lin` -> rename keys when loading.
3. train.py    : new PyG `Data` objects are not hashable, so `set(X) - set(X_test)` fails
                 -> build the training list by index (same samples, same original order).
"""
import argparse
import os
import re


def patch_model(path):
    s = open(path).read()
    s = s.replace("data.node_attrs", "data['node_attrs']")
    open(path, "w").write(s)


def patch_evaluate(path):
    s = open(path).read()
    if "lin_src" in s:
        return
    pat = re.compile(
        r"^([ \t]*)model\.load_state_dict\(torch\.load\(f'\{direc\}/\{pt\}'\)\)", re.M
    )
    assert pat.search(s), "could not find the load_state_dict line in evaluate.py"

    def rep(m):
        i = m.group(1)
        return (
            i + "sd = torch.load(f'{direc}/{pt}', map_location='cpu')\n"
            + i + "sd = {k.replace('lin_src', 'lin'): v for k, v in sd.items() if 'lin_dst' not in k}\n"
            + i + "model.load_state_dict(sd)"
        )

    open(path, "w").write(pat.sub(rep, s))


def patch_train(path):
    s = open(path).read()
    old = "X_train = list(set(X) - set(X_test))"
    new = "X_train = [d for i, d in enumerate(X) if i < test_sta or i > test_end]"
    if old in s:
        open(path, "w").write(s.replace(old, new))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--upstream", default="EpiGraph", help="path to the cloned upstream repo")
    a = ap.parse_args()
    patch_model(os.path.join(a.upstream, "model.py"))
    patch_evaluate(os.path.join(a.upstream, "evaluate.py"))
    patch_train(os.path.join(a.upstream, "train.py"))
    print("upstream patched OK:", a.upstream)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/src/train_seeded.py
"""Run the upstream train.py with the *PyTorch* random number generators seeded.

Upstream train.py only seeds Python's `random` (which controls the order of the 200
antigens before the consecutive 10-fold split, `--seed`, default 30). It never seeds
PyTorch, so the initial weights of every model differ from run to run.

This wrapper seeds torch / numpy first and then executes the unmodified train.py, so
that two sources of randomness can be controlled independently:

    --torch_seed N   weight initialisation (and any torch RNG use)
    --seed N         fold split (upstream argument, forwarded untouched)

Usage (all other arguments are forwarded to train.py):

    python src/train_seeded.py --upstream EpiGraph --torch_seed 1 --seed 30 \
        --model_save_dir runs/init_s1/models

Note: even with identical seeds, GPU runs are not bit-for-bit reproducible, because
scatter/atomic-add kernels used by GATConv are non-deterministic. Seeds make runs
*comparable* (same initialisation, same split), not identical.
"""
import argparse
import os
import runpy
import sys

import numpy as np
import torch


def main():
    ap = argparse.ArgumentParser(add_help=False)
    ap.add_argument("--upstream", required=True, help="path to the (patched) upstream repo")
    ap.add_argument("--torch_seed", type=int, required=True)
    known, rest = ap.parse_known_args()

    upstream = os.path.abspath(known.upstream)
    # make relative --model_save_dir independent of the chdir below
    for i, tok in enumerate(rest):
        if tok == "--model_save_dir" and i + 1 < len(rest):
            rest[i + 1] = os.path.abspath(rest[i + 1])

    torch.manual_seed(known.torch_seed)
    torch.cuda.manual_seed_all(known.torch_seed)
    np.random.seed(known.torch_seed)

    # train.py reads the .pkl files and imports `model` relative to the repo root
    os.chdir(upstream)
    sys.path.insert(0, upstream)
    sys.argv = ["train.py"] + rest
    runpy.run_path("train.py", run_name="__main__")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/src/eval_ensemble.py
"""Evaluate an ensemble of EpiGraph checkpoints on the epitope3D test set.

Differs from upstream evaluate.py in three ways (the ensemble definition is the same:
mean of the per-model probabilities over surface residues, fixed decision threshold):

  * metrics are written to JSON with full precision (upstream prints 2 decimals);
  * metrics of every single model are reported next to the ensemble;
  * a bootstrap over *antigens* gives 95% confidence intervals for the ensemble.

    python src/eval_ensemble.py --upstream EpiGraph --ckpt_dir runs/init_s1/models \
        --out runs/init_s1/metrics.json

Requires the upstream repo to be patched first (src/patch_upstream.py).
"""
import argparse
import json
import os
import pickle
import sys

import numpy as np
import torch
from sklearn import metrics as skm


def load_state(path):
    sd = torch.load(path, map_location="cpu")
    # checkpoints written by an old torch_geometric use lin_src/lin_dst
    return {k.replace("lin_src", "lin"): v for k, v in sd.items() if "lin_dst" not in k}


@torch.no_grad()
def predict_all(model, data_list, device):
    """Per-antigen arrays of predicted probability on surface residues (train_mask)."""
    model.eval()
    out = []
    for d in data_list:
        d = d.to(device)
        p = model(d)[d.train_mask].reshape(-1).cpu().numpy()
        out.append(p)
    return out


def threshold_free(y, p):
    fpr, tpr, _ = skm.roc_curve(y, p)
    prec, rec, _ = skm.precision_recall_curve(y, p)
    return {
        "auc_roc": float(skm.auc(fpr, tpr)),
        # same definition as upstream evaluate.py (trapezoid under the PR curve)
        "auc_pr": float(skm.auc(rec, prec)),
        "avg_precision": float(skm.average_precision_score(y, p)),
    }


def all_metrics(y, p, thr):
    m = threshold_free(y, p)
    yhat = (p > thr).astype(int)
    m["f1"] = float(skm.f1_score(y, yhat))
    m["bacc"] = float(skm.balanced_accuracy_score(y, yhat))
    m["mcc"] = float(skm.matthews_corrcoef(y, yhat))
    return m


def summarize(values):
    v = np.asarray(values, dtype=float)
    return {
        "mean": float(v.mean()),
        "std": float(v.std(ddof=1)) if len(v) > 1 else 0.0,
        "min": float(v.min()),
        "max": float(v.max()),
        "values": [float(x) for x in v],
    }


def bootstrap_ci(y_list, p_list, thr, n_boot, seed):
    """Resample antigens (not residues) with replacement; residues of one antigen are correlated."""
    rng = np.random.default_rng(seed)
    n = len(y_list)
    keys = ["auc_roc", "auc_pr", "f1", "bacc", "mcc"]
    draws = {k: [] for k in keys}
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        y = np.concatenate([y_list[i] for i in idx])
        p = np.concatenate([p_list[i] for i in idx])
        if y.min() == y.max():
            continue
        m = all_metrics(y, p, thr)
        for k in keys:
            draws[k].append(m[k])
    return {k: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] for k, v in draws.items()}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--upstream", required=True)
    ap.add_argument("--ckpt_dir", required=True)
    ap.add_argument("--out", required=True, help="metrics JSON to write")
    ap.add_argument("--threshold", type=float, default=0.1481, help="upstream default")
    ap.add_argument("--hid_dim", type=int, default=128)
    ap.add_argument("--num_head", type=int, default=8)
    ap.add_argument("--n_boot", type=int, default=1000)
    ap.add_argument("--boot_seed", type=int, default=0)
    ap.add_argument("--device", default="cuda")
    a = ap.parse_args()

    sys.path.insert(0, os.path.abspath(a.upstream))
    from model import GAT  # noqa: E402  (needs the patched upstream model.py)

    device = torch.device(a.device if torch.cuda.is_available() else "cpu")
    with open(os.path.join(a.upstream, "epitope3d_test_data.pkl"), "rb") as f:
        test = pickle.load(f)

    ckpts = sorted(f for f in os.listdir(a.ckpt_dir) if f.endswith(".pt"))
    assert ckpts, f"no .pt files in {a.ckpt_dir}"

    y_list = [d.y[d.train_mask].cpu().numpy().astype(int) for d in test]
    y = np.concatenate(y_list)

    per_model = []  # list over models of list over antigens
    for name in ckpts:
        model = GAT(in_dim=1792, hid_dim=a.hid_dim, out_dim=1, num_head=a.num_head, out_head=1)
        model.load_state_dict(load_state(os.path.join(a.ckpt_dir, name)))
        model.to(device)
        per_model.append(predict_all(model, test, device))

    ens_list = [np.mean([pm[i] for pm in per_model], axis=0) for i in range(len(test))]
    p_ens = np.concatenate(ens_list)

    single = [all_metrics(y, np.concatenate(pm), a.threshold) for pm in per_model]
    keys = ["auc_roc", "auc_pr", "f1", "bacc", "mcc"]

    per_antigen = [
        skm.roc_auc_score(yy, pp) for yy, pp in zip(y_list, ens_list) if 0 < yy.sum() < len(yy)
    ]

    result = {
        "ckpt_dir": a.ckpt_dir,
        "checkpoints": ckpts,
        "n_models": len(ckpts),
        "n_antigens": len(test),
        "n_surface_residues": int(len(y)),
        "positive_rate": float(y.mean()),
        "threshold": a.threshold,
        "ensemble": all_metrics(y, p_ens, a.threshold),
        "ensemble_ci95_bootstrap_antigens": bootstrap_ci(
            y_list, ens_list, a.threshold, a.n_boot, a.boot_seed
        ),
        "single_models": {k: summarize([m[k] for m in single]) for k in keys},
        "per_antigen_auc_roc": {
            "n_antigens_with_both_classes": len(per_antigen),
            "mean": float(np.mean(per_antigen)),
            "median": float(np.median(per_antigen)),
        },
    }

    os.makedirs(os.path.dirname(os.path.abspath(a.out)), exist_ok=True)
    with open(a.out, "w") as f:
        json.dump(result, f, indent=2)
    np.savez_compressed(
        os.path.splitext(a.out)[0] + "_preds.npz",
        y=y,
        antigen=np.concatenate([np.full(len(v), i) for i, v in enumerate(y_list)]),
        ensemble=p_ens,
        per_model=np.stack([np.concatenate(pm) for pm in per_model]),
    )

    e = result["ensemble"]
    ci = result["ensemble_ci95_bootstrap_antigens"]
    print(f"[{a.ckpt_dir}] {len(ckpts)} models, {len(test)} antigens, "
          f"{len(y)} surface residues, positive rate {y.mean():.3f}")
    for k in keys:
        print(f"  {k:8s} {e[k]:.4f}   95% CI [{ci[k][0]:.3f}, {ci[k][1]:.3f}]   "
              f"single-model mean {result['single_models'][k]['mean']:.4f} "
              f"± {result['single_models'][k]['std']:.4f}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/src/aggregate.py
"""Aggregate the per-run metrics written by eval_ensemble.py into summary tables.

Expected layout (created by notebooks/02_multiseed_robustness_kaggle.ipynb):

    runs/pretrained/metrics.json              author checkpoints
    runs/init_s<N>/metrics.json (+ train.log) experiment A: only weight init differs, split seed 30
    runs/split_s<N>/metrics.json (+ train.log) experiment B: fold split and init both differ

    python src/aggregate.py --runs runs --out results
"""
import argparse
import glob
import json
import os
import re

import numpy as np
from scipy import stats

KEYS = ["auc_roc", "auc_pr", "f1", "bacc", "mcc"]
LABEL = {"auc_roc": "AUC-ROC", "auc_pr": "AUC-PR", "f1": "F1", "bacc": "BACC", "mcc": "MCC"}
# Paper (Choi & Kim, Sci Rep 2024, 14:27496), Table 1 / Table 3, epitope3D test set
PAPER = {"auc_roc": 0.73, "auc_pr": 0.23, "f1": 0.29, "bacc": 0.62, "mcc": 0.19}
PAPER_ROUND = 0.005  # paper values are rounded to 2 decimals


def load_runs(runs_dir):
    runs = {}
    for path in sorted(glob.glob(os.path.join(runs_dir, "*", "metrics.json"))):
        name = os.path.basename(os.path.dirname(path))
        with open(path) as f:
            runs[name] = json.load(f)
        log = os.path.join(os.path.dirname(path), "train.log")
        if os.path.exists(log):
            runs[name]["_train"] = parse_train_log(log)
    return runs


def parse_train_log(path):
    txt = open(path, errors="ignore").read()
    ep = [int(x) for x in re.findall(r"The best epoch: (\d+)", txt)]
    vl = [float(x) for x in re.findall(r"The best val loss: ([0-9.]+)", txt)]
    va = [float(x) for x in re.findall(r"The best auc-roc: ([0-9.]+)", txt)]
    return {"best_epoch": ep, "best_val_loss": vl, "best_val_auc": va}


def ci_t(values, level=0.95):
    v = np.asarray(values, float)
    n = len(v)
    if n < 2:
        return float("nan"), float("nan")
    h = stats.t.ppf(0.5 + level / 2, n - 1) * v.std(ddof=1) / np.sqrt(n)
    return v.mean() - h, v.mean() + h


def fmt(x, d=3):
    return "n/a" if x is None or (isinstance(x, float) and np.isnan(x)) else f"{x:.{d}f}"


def group_table(runs, names, title):
    out = [f"### {title}", ""]
    if not names:
        return out + ["_no runs found_", ""]
    out += ["| Run | " + " | ".join(LABEL[k] for k in KEYS) + " |", "|---|" + "---|" * len(KEYS)]
    for n in names:
        out.append(f"| {n} | " + " | ".join(fmt(runs[n]["ensemble"][k]) for k in KEYS) + " |")
    vals = {k: [runs[n]["ensemble"][k] for n in names] for k in KEYS}
    out.append("| **mean ± std** | " + " | ".join(
        f"**{np.mean(vals[k]):.3f} ± {np.std(vals[k], ddof=1) if len(names) > 1 else 0:.3f}**" for k in KEYS) + " |")
    out.append("| min – max | " + " | ".join(f"{min(vals[k]):.3f} – {max(vals[k]):.3f}" for k in KEYS) + " |")
    cis = {k: ci_t(vals[k]) for k in KEYS}
    out.append(f"| 95% t-interval of the mean (n={len(names)}) | " + " | ".join(
        f"[{fmt(cis[k][0])}, {fmt(cis[k][1])}]" for k in KEYS) + " |")
    out.append("| Paper value | " + " | ".join(f"{PAPER[k]:.2f}" for k in KEYS) + " |")
    verdict = []
    for k in KEYS:
        lo, hi = cis[k]
        if np.isnan(lo):
            verdict.append("n/a")
        else:
            verdict.append("inside" if lo - PAPER_ROUND <= PAPER[k] <= hi + PAPER_ROUND else "OUTSIDE")
    out.append("| Paper value within interval (±0.005 rounding)? | " + " | ".join(verdict) + " |")
    out.append("")

    # single models pooled over the group
    out += ["Single (non-ensembled) models, pooled over all folds of the group:", "",
            "| Metric | mean ± std over models | ensemble mean | ensemble gain |", "|---|---|---|---|"]
    for k in KEYS:
        allv = np.concatenate([runs[n]["single_models"][k]["values"] for n in names])
        out.append(f"| {LABEL[k]} | {allv.mean():.3f} ± {allv.std(ddof=1):.3f} (n={len(allv)}) | "
                   f"{np.mean(vals[k]):.3f} | {np.mean(vals[k]) - allv.mean():+.3f} |")
    out.append("")

    tr = [runs[n]["_train"] for n in names if "_train" in runs[n]]
    if tr:
        auc = [np.mean(t["best_val_auc"]) for t in tr if t["best_val_auc"]]
        ep = np.concatenate([t["best_epoch"] for t in tr if t["best_epoch"]])
        out.append(f"Validation side: mean best val AUC per run = {np.mean(auc):.3f} ± "
                   f"{np.std(auc, ddof=1) if len(auc) > 1 else 0:.3f}; best epoch median {np.median(ep):.0f} "
                   f"(range {ep.min()}–{ep.max()}, n={len(ep)} folds).")
        out.append("")
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--runs", default="runs")
    ap.add_argument("--out", default="results")
    a = ap.parse_args()
    runs = load_runs(a.runs)
    os.makedirs(a.out, exist_ok=True)

    md = ["# Multi-seed robustness summary (auto-generated by src/aggregate.py)", ""]
    if "pretrained" in runs:
        r = runs["pretrained"]
        ci = r["ensemble_ci95_bootstrap_antigens"]
        md += [f"## Author checkpoints on the test set ({r['n_antigens']} antigens, "
               f"{r['n_surface_residues']} surface residues, {100 * r['positive_rate']:.1f}% epitope)", "",
               "| Metric | Paper | Ensemble (10 models) | 95% bootstrap CI over antigens | Single-model mean ± std |",
               "|---|---|---|---|---|"]
        for k in KEYS:
            s = r["single_models"][k]
            md.append(f"| {LABEL[k]} | {PAPER[k]:.2f} | {r['ensemble'][k]:.3f} | "
                      f"[{ci[k][0]:.3f}, {ci[k][1]:.3f}] | {s['mean']:.3f} ± {s['std']:.3f} |")
        pa = r["per_antigen_auc_roc"]
        md += ["", f"Per-antigen AUC-ROC (ensemble): mean {pa['mean']:.3f}, median {pa['median']:.3f} "
                   f"over {pa['n_antigens_with_both_classes']} antigens.", ""]

    A = [n for n in runs if n.startswith("init_s")]
    B = [n for n in runs if n.startswith("split_s")]
    md += ["## Retrained models", ""]
    md += group_table(runs, A, "Experiment A - weight-initialisation seeds only (fold split fixed, seed 30)")
    md += group_table(runs, B, "Experiment B - fold split and initialisation both re-randomised")
    if "pretrained" in runs and (A or B):
        md += ["## Retrained vs. author checkpoints (ensemble AUC-ROC)", ""]
        pre = runs["pretrained"]["ensemble"]["auc_roc"]
        for title, g in (("A", A), ("B", B)):
            if g:
                v = [runs[n]["ensemble"]["auc_roc"] for n in g]
                md.append(f"- Experiment {title}: mean {np.mean(v):.3f} vs. author checkpoints {pre:.3f} "
                          f"(difference {np.mean(v) - pre:+.3f}).")
        md.append("")

    text = "\n".join(md)
    with open(os.path.join(a.out, "summary.md"), "w", encoding="utf-8") as f:
        f.write(text)
    with open(os.path.join(a.out, "summary.json"), "w") as f:
        json.dump({n: {k: v for k, v in r.items() if k != "_train"} for n, r in runs.items()}, f, indent=2)
    print(text)


if __name__ == "__main__":
    main()

In [ ]:
# ---- 2. patch upstream for the new torch_geometric (idempotent) ---------------------
!python /kaggle/working/src/patch_upstream.py --upstream /kaggle/working/EpiGraph

In [ ]:
# ---- 3. author checkpoints: 4-decimal metrics + bootstrap CI ------------------------
import os, subprocess, time, json
W = "/kaggle/working"; RUNS = f"{W}/runs"
os.makedirs(f"{RUNS}/pretrained", exist_ok=True)

def sh(cmd, log=None):
    t = time.time()
    if log:
        with open(log, "w") as f:
            r = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT)
    else:
        r = subprocess.run(cmd)
    assert r.returncode == 0, f"failed: {' '.join(cmd)} (see {log})"
    return time.time() - t

def evaluate(name, ckpt_dir):
    sh(["python", f"{W}/src/eval_ensemble.py", "--upstream", f"{W}/EpiGraph",
        "--ckpt_dir", ckpt_dir, "--out", f"{RUNS}/{name}/metrics.json"])

if not os.path.exists(f"{RUNS}/pretrained/metrics.json"):
    evaluate("pretrained", f"{W}/EpiGraph/checkpoint")

In [ ]:
# ---- 4. retrain (resumable) ------------------------------------------------------
import shutil, glob
jobs = [(f"init_s{s}", s, BASE_SPLIT_SEED) for s in INIT_SEEDS] + [(f"split_s{s}", s, s) for s in SPLIT_SEEDS]

for name, torch_seed, split_seed in jobs:
    run = f"{RUNS}/{name}"
    if os.path.exists(f"{run}/metrics.json"):
        print(name, "already done, skipping"); continue
    shutil.rmtree(run, ignore_errors=True)       # drop half-finished runs
    os.makedirs(run)
    dt = sh(["python", "-u", f"{W}/src/train_seeded.py", "--upstream", f"{W}/EpiGraph",
             "--torch_seed", str(torch_seed), "--seed", str(split_seed), "--epochs", str(EPOCHS),
             "--model_save_dir", f"{run}/models"], log=f"{run}/train.log")
    assert len(glob.glob(f"{run}/models/*.pt")) == 10, "expected 10 checkpoints"
    evaluate(name, f"{run}/models")
    if not KEEP_CHECKPOINTS:
        shutil.rmtree(f"{run}/models")
    m = json.load(open(f"{run}/metrics.json"))["ensemble"]
    print(f"{name}: train {dt/60:.1f} min | AUC-ROC {m['auc_roc']:.4f} AUC-PR {m['auc_pr']:.4f} "
          f"F1 {m['f1']:.4f} BACC {m['bacc']:.4f} MCC {m['mcc']:.4f}")

In [ ]:
# ---- 5. aggregate + package small outputs ------------------------------------------
!python /kaggle/working/src/aggregate.py --runs /kaggle/working/runs --out /kaggle/working/results
!cd /kaggle/working && zip -qr results.zip results runs -x "runs/*/models/*" && ls -la results.zip